In [ ]:
import pandas as pd
from utils.utils import setup_model, pull_spectral_images, save_model
from sklearn.model_selection import train_test_split
from net.generator import SpectralImageDataGenerator
from net.net_def import create_3d_autoencoder

# Load the TensorBoard notebook extension
%load_ext tensorboard
import tensorflow as tf


# Model Configuration
MODEL_NAME = '3d_autoencoder'
INPUT_FILES = "hyspecnet-11k/patches_reduced"
INPUT_FILE_TYPE = 'npy'
BATCH_SIZE = 32
EPOCHS = 2
USE_CALLBACKS = ['tf_logs', 'keras_checkpoints', 'early_stopping']

In [ ]:
# Load the spectral images
spectral_images, input_shape = pull_spectral_images(INPUT_FILES, type=INPUT_FILE_TYPE)
print(f"Loaded {len(spectral_images)} spectral images with input shape {input_shape}.")

In [ ]:
# Split the data into training and validation sets
image_paths_train, image_paths_val = train_test_split(spectral_images, test_size=0.2, random_state=42)
train_gen = SpectralImageDataGenerator(image_paths_train, batch_size=BATCH_SIZE)
val_gen = SpectralImageDataGenerator(image_paths_val, batch_size=BATCH_SIZE)
print(f"Inputs set to {input_shape} with {len(image_paths_train)} training images and {len(image_paths_val)} validation images.")

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.legacy import SGD

# Define the model
autoencoder = create_3d_autoencoder(input_shape)
#autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
autoencoder.compile(optimizer=SGD(learning_rate=0.01, momentum=0.9), loss='mean_squared_error', metrics=['mae'])
autoencoder.summary()

In [5]:
# Setup the model directory and callbacks
model_dir, callbacks = setup_model(MODEL_NAME, use_callbacks=USE_CALLBACKS)
print(f"Using callbacks: {callbacks}")

%tensorboard --logdir logs/fit

# Train the model
print(f"Training model {MODEL_NAME} with input shape {input_shape} and {len(image_paths_train)} images.")
history = autoencoder.fit(
    train_gen,  # The generator provides both x and y
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True, 
    validation_data=val_gen, 
    verbose='auto',
    callbacks=callbacks
)

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation loss values
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Plot training & validation MAE values
plt.plot(history.history['mae'])
plt.plot(history.history['val_mae'])
plt.title('Model MAE')
plt.ylabel('MAE')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# Save the model
saved_model = save_model(autoencoder, MODEL_NAME, model_dir)  
print(f"Model and accompanying info saved to {model_dir}")